# Important Note:

Depending on the way the file is run, you may run out of space on your Google Drive. Specifically, the `llama_features.pkl` file is extremely large (around 30 GB), and the K / alpha sweep files can take a similar amount of space.

If the \{path\} and \{path_sm\} references are kept as they are, there should be no issue on this front.

Similarly, since this requires a lot of compute, there is a chance that you may run out of RAM if your Google Colab is unable to run on an A100 GPU.

# Installation / Setup

This cell is optional to run - only do so if you plan on performing the experiments yourself and need to export files to Google Drive.

If you do, **make sure you've created the following things:**

1. A folder titled "company_similarity_sae" in "MyDrive".
2. Two folderes within this folder: one titled "data" and one titled "figures".

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
!pip install huggingface_hub
!pip install tabulate datasets
!pip install cupy-cuda12x
from google.colab import userdata

If you do not already have a HuggingFace token, you can acquire one on the HuggingFace website. Once you hvae done this, you can add a "Secret" using the key icon on the left menu bar. Make sure you title the Secret "HF_TOKEN", and paste the token into the "Value" section. Also, ensure that "Notebook access" is checked.

**This is not necessary to run. Only do so if you find yourself having permission errors with any of the HuggingFace code.**

In [ ]:
from huggingface_hub import login
HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

In [ ]:
!git clone https://github.com/danielliu5670/WSFS-SAF

Cloning into 'WSFS-SAF'...
remote: Enumerating objects: 90, done.
remote: Counting objects: 100% (90/90), done.
remote: Compressing objects: 100% (66/66), done.
remote: Total 90 (delta 37), reused 68 (delta 21), pack-reused 0 (from 0)
Receiving objects: 100% (90/90), 739.17 KiB | 46.20 MiB/s, done.
Resolving deltas: 100% (37/37), done.


This cell pulls the pre-computed `llama_features.pkl` file from our HuggingFace repository. We were unable to upload it to GitHub, since it was far too large. If you export your own version (in the Feature Collection section), you may have to adjust all of the references to `{path}`.

In [ ]:
from huggingface_hub import hf_hub_download

path = hf_hub_download(
    repo_id="danielliu1/liu_mesyngier_company_similarity_sae",
    filename="llama_features.pkl",
    repo_type="dataset"
)

llama_features.pkl:   0%|          | 0.00/29.4G [00:00<?, ?B/s]

This cell does the same, but for the `llama_selection_model.pkl` file.

In [ ]:
path_sm = hf_hub_download(
    repo_id="danielliu1/liu_mesyngier_company_similarity_sae",
    filename="llama_selection_model.pkl",
    repo_type="dataset"
)

llama_selection_model.pkl:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

In [ ]:
from datasets import load_dataset
import os

cov_path = "/content/cov_dataset.parquet"
pairs_path = "/content/pairs_dataset.parquet"

if not os.path.exists(cov_path):
    load_dataset(
        "Mateusz1017/annual_reports_tokenized_llama3_logged_returns"
        "_no_null_returns_and_incomplete_descriptions_24k"
    )["train"].to_pandas().to_parquet(cov_path)

if not os.path.exists(pairs_path):
    load_dataset("v1ctor10/cos_sim_4000pca_exp")["train"].to_pandas().to_parquet(pairs_path)

README.md:   0%|          | 0.00/697 [00:00<?, ?B/s]

data/train-00000-of-00005.parquet:   0%|          | 0.00/140M [00:00<?, ?B/s]

data/train-00001-of-00005.parquet:   0%|          | 0.00/154M [00:00<?, ?B/s]

data/train-00002-of-00005.parquet:   0%|          | 0.00/158M [00:00<?, ?B/s]

data/train-00003-of-00005.parquet:   0%|          | 0.00/155M [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/27888 [00:00<?, ? examples/s]

README.md:   0%|          | 0.00/592 [00:00<?, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/320M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/325M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15002324 [00:00<?, ? examples/s]

# Feature Collection

This code extracts all of the features we will need from the parent paper's code.

In [ ]:
from datasets import load_dataset
ds = load_dataset("marco-molinari/company_reports_with_features")
df = ds["train"].to_pandas()[["__index_level_0__", "features"]]
df.to_pickle("/content/drive/MyDrive/company_similarity_sae/data/llama_features.pkl")

In [ ]:
from datasets import load_dataset
ds = load_dataset("marco-molinari/company_reports_with_features", streaming=True)
row = next(iter(ds["train"]))
print(row["__index_level_0__"])

Resolving data files:   0%|          | 0/59 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/59 [00:00<?, ?it/s]

0


# Main

These next two code cells iterate through all of our K and alpha values (as mentioned in the paper), and show their correlation-at-k performance.

Note that they do not output the parameter files to Google Drive - we will only do so for the K = 1000 and alpha = 0 configuration. This is for space-saving reasons. If you would like to still output them, you can add the --out_pairs command that you will see later.

In [ ]:
!python WSFS-SAF/extract_features.py \
    --features-pkl {path} \
    --top-k 500 750 1000 1250 1500 \
    --norm-alpha 0 0.25 0.5 0.75 1.0 \
    --score-weight \
    --cov-ds /content/cov_dataset.parquet \
    --original-pairs-ds /content/pairs_dataset.parquet \
    --out-model /content/drive/MyDrive/company_similarity_sae/data/llama_selection_model.pkl # This argument can be commented out if there was no Google Drive mounting.

README.md: 100% 697/697 [00:00<00:00, 2.52MB/s]
data/train-00000-of-00005.parquet: 100% 140M/140M [00:01<00:00, 76.6MB/s]
data/train-00001-of-00005.parquet: 100% 154M/154M [00:02<00:00, 76.6MB/s]
data/train-00002-of-00005.parquet: 100% 158M/158M [00:03<00:00, 41.4MB/s]
data/train-00003-of-00005.parquet: 100% 155M/155M [00:01<00:00, 110MB/s]
data/train-00004-of-00005.parquet: 100% 159M/159M [00:01<00:00, 87.7MB/s]
Generating train split: 100% 27888/27888 [00:05<00:00, 4942.77 examples/s]
README.md: 100% 592/592 [00:00<00:00, 346kB/s]
data/train-00000-of-00002.parquet: 100% 320M/320M [00:02<00:00, 114MB/s]
data/train-00001-of-00002.parquet: 100% 325M/325M [00:02<00:00, 116MB/s]
Generating train split: 100% 15002324/15002324 [00:07<00:00, 2096306.76 examples/s]
Scoring features (GPU): 100% 256/256 [01:44<00:00,  2.44it/s]

top-k = 500

┌───────┬──────────────┬──────────────────────┬──────────────────────┐
│ alpha │ Spearman rho │ Precision-at-k (all) │ Precision-at-k (OOS) │
├───────┼────

In [ ]:
!python WSFS-SAF/extract_features.py \
    --features-pkl {path} \
    --load-model {path_sm} \
    --top-k 1750 2000 2250 2500 \
    --norm-alpha 0 0.25 0.5 0.75 1.0 \
    --score-weight \
    --cov-ds /content/cov_dataset.parquet \
    --original-pairs-ds /content/pairs_dataset.parquet \
    --out-model /content/drive/MyDrive/company_similarity_sae/data/llama_selection_model.pkl # This argument can be commented out if there was no Google Drive mounting.


top-k = 1750

┌───────┬──────────────┬──────────────────────┬──────────────────────┐
│ alpha │ Spearman rho │ Precision-at-k (all) │ Precision-at-k (OOS) │
├───────┼──────────────┼──────────────────────┼──────────────────────┤
│ 0.0   │ 0.1833       │ Top   0.5% = 0.3671  │ Top   0.5% = 0.4050  │
│       │              │ Top   1.0% = 0.3427  │ Top   1.0% = 0.3787  │
│       │              │ Top   2.0% = 0.3212  │ Top   2.0% = 0.3519  │
│       │              │ Top   5.0% = 0.2955  │ Top   5.0% = 0.3197  │
│       │              │ Top  10.0% = 0.2758  │ Top  10.0% = 0.2954  │
├───────┼──────────────┼──────────────────────┼──────────────────────┤
│ 0.25  │ 0.1825       │ Top   0.5% = 0.3638  │ Top   0.5% = 0.4008  │
│       │              │ Top   1.0% = 0.3392  │ Top   1.0% = 0.3750  │
│       │              │ Top   2.0% = 0.3187  │ Top   2.0% = 0.3482  │
│       │              │ Top   5.0% = 0.2944  │ Top   5.0% = 0.3169  │
│       │              │ Top  10.0% = 0.2753  │ Top  10.0% = 0

This code, as previously stated, saves the configuration files for K = 2000 and alpha = 0.

In [ ]:
!python WSFS-SAF/extract_features.py \
    --features-pkl {path} \
    --load-model {path_sm} \
    --top-k 2000 \
    --norm-alpha 0 \
    --score-weight \
    --cov-ds /content/cov_dataset.parquet \
    --original-pairs-ds /content/pairs_dataset.parquet \
    --out-pairs /content/drive/MyDrive/company_similarity_sae/data/llama_pairs_sw.pkl \
    --out-model /content/drive/MyDrive/company_similarity_sae/data/llama_selection_model.pkl # This and the previous argument can be commented out if there was no Google Drive mounting.


top-k = 2000

┌───────┬──────────────┬──────────────────────┬──────────────────────┐
│ alpha │ Spearman rho │ Precision-at-k (all) │ Precision-at-k (OOS) │
├───────┼──────────────┼──────────────────────┼──────────────────────┤
│ 0.0   │ 0.1830       │ Top   0.5% = 0.3673  │ Top   0.5% = 0.4053  │
│       │              │ Top   1.0% = 0.3429  │ Top   1.0% = 0.3789  │
│       │              │ Top   2.0% = 0.3219  │ Top   2.0% = 0.3522  │
│       │              │ Top   5.0% = 0.2963  │ Top   5.0% = 0.3206  │
│       │              │ Top  10.0% = 0.2764  │ Top  10.0% = 0.2963  │
└───────┴──────────────┴──────────────────────┴──────────────────────┘

Summary:

┌───────┬───────┬────────┬────────────────┐
│ top k │ alpha │ rho    │ top 1.0% (OOS) │
├───────┼───────┼────────┼────────────────┤
│ 2000  │ 0.00  │ 0.1830 │ 0.3789         │
├───────┼───────┼────────┼────────────────┤
│ SIC   │       │ 0.0345 │ 0.2839         │
└───────┴───────┴────────┴────────────────┘


# Final Results (Pairwise)

This section prints out a graph that compares our approach with the parent paper's approach and the SIC code baseline.

In [ ]:
!python WSFS-SAF/evaluate.py \
    --features-pkl {path} \
    --load-model {path_sm} \
    --top-k 2000 \
    --norm-alpha 0 \
    --cov-ds /content/cov_dataset.parquet \
    --original-pairs-ds /content/pairs_dataset.parquet


Spearman rho (OOS)
┌──────────────────────────────┬────────────────┬───────────┐
│ Approach                     │   Spearman rho │   p-value │
├──────────────────────────────┼────────────────┼───────────┤
│ New approach (k=2000, α=0.0) │         0.2204 │         0 │
│ Parent paper (PCA 4000-dim)  │         0.0278 │         0 │
└──────────────────────────────┴────────────────┴───────────┘

Correlation-at-k, OOS 2014-2020
┌───────────┬────────────────┬────────────────┬────────────────┐
│ Cutoff    │   New approach │   Parent paper │ SIC baseline   │
├───────────┼────────────────┼────────────────┼────────────────┤
│ top 0.5%  │         0.4053 │         0.1592 │                │
│ top 1.0%  │         0.3789 │         0.1598 │ 0.2835         │
│ top 2.0%  │         0.3522 │         0.1755 │                │
│ top 5.0%  │         0.3206 │         0.1832 │                │
│ top 10.0% │         0.2963 │         0.1798 │                │
└───────────┴────────────────┴────────────────┴────────

# Final Results (Clustering)

This section compares our approach with the parent paper's and SIC code approaches, but specifically using the MC(Gk) clustering evaluation metric.

In [ ]:
!python WSFS-SAF/evaluate_clustering.py \
    --features-pkl {path} \
    --load-model {path_sm} \
    --top-k 2000 \
    --norm-alpha 0 \
    --cov-ds /content/cov_dataset.parquet \
    --original-pairs-ds /content/pairs_dataset.parquet


Parent paper:
Building MSTs: 100% 25/25 [01:27<00:00,  3.52s/it]
Sweeping Theta: 100% 36/36 [00:46<00:00,  1.30s/it]

New method:
Building MSTs: 100% 25/25 [01:21<00:00,  3.27s/it]
Sweeping Theta: 100% 36/36 [00:19<00:00,  1.83it/s]

Unweighted MC(Gk)
┌─────────────────┬────────┬─────────────┐
│ Approach        │    OOS │   All years │
├─────────────────┼────────┼─────────────┤
│ New method      │ 0.293  │      0.302  │
│ Parent paper    │ 0.3789 │      0.3733 │
│ SIC code        │ 0.2429 │      0.2311 │
│ Population mean │ 0.1609 │      0.1609 │
└─────────────────┴────────┴─────────────┘

Weighted MC(Gk)
┌─────────────────┬────────┬─────────────┐
│ Approach        │    OOS │   All years │
├─────────────────┼────────┼─────────────┤
│ New method      │ 0.2172 │      0.224  │
│ Parent paper    │ 0.1612 │      0.1513 │
│ SIC code        │ 0.2771 │      0.252  │
│ Population mean │ 0.1609 │      0.1609 │
└─────────────────┴────────┴─────────────┘

New method OOS (θ=-1.60):
┌────────┬─────

# Ablation

This last section performs all of the ablation tests discussed in the paper.

This is the first ablation test - "WSFS-SAF w/o Supervised Feature Selection".

In [ ]:
!python WSFS-SAF/ablation/ablation_unsupervised_sae.py \
    --features-pkl {path} \
    --cov-ds /content/cov_dataset.parquet \
    --original-pairs-ds /content/pairs_dataset.parquet \
    --sae-spearman-oos 0.2204 \
    --sae-spearman-all 0.1830 \
    --sae-lifts-oos "0.4053,0.3789,0.3522,0.3206,0.2963" \
    --sae-lifts-all "0.3673,0.3429,0.3219,0.2963,0.2764" \
    --parent-spearman-all 0.0217 \
    --parent-lifts-oos "0.1592,0.1598,0.1755,0.1832,0.1798" \
    --parent-spearman-oos 0.0278 \
    --parent-lifts-all "0.1415,0.1445,0.1549,0.1665,0.1666"


Unsupervised SAE Results (OOS):
┌──────────────────┬────────────────┬───────────┬────────────────────┐
│ Method           │ Spearman rho   │ Cutoff    │ Mean correlation   │
├──────────────────┼────────────────┼───────────┼────────────────────┤
│ Unsupervised SAE │ 0.0502         │ top 0.5%  │ 0.4316             │
│ (cosine)         │                │ top 1.0%  │ 0.3801             │
│                  │                │ top 2.0%  │ 0.3285             │
│                  │                │ top 5.0%  │ 0.2675             │
│                  │                │ top 10.0% │ 0.2294             │
├──────────────────┼────────────────┼───────────┼────────────────────┤
│ Unsupervised SAE │ 0.0372         │ top 0.5%  │ 0.2005             │
│ (dot product)    │                │ top 1.0%  │ 0.2039             │
│                  │                │ top 2.0%  │ 0.2028             │
│                  │                │ top 5.0%  │ 0.1962             │
│                  │                │ top 10

This is the second ablation test - "WSFS-SAF w/ Principal Component Analysis".

In [ ]:
!python WSFS-SAF/ablation/ablation_pca_supervised.py \
    --features-pkl {path} \
    --cov-ds /content/cov_dataset.parquet \
    --original-pairs-ds /content/pairs_dataset.parquet \
    --sae-spearman-oos 0.2204 \
    --sae-lifts-oos "0.4053,0.3789,0.3522,0.3206,0.2963" \
    --top-k 2000


Supervised PCA Results (OOS 2014-2020, 6,181,505 pairs):
┌────────────────┬────────────────┬───────────┬────────────────────┐
│ Method         │ Spearman rho   │ Cutoff    │ Mean correlation   │
├────────────────┼────────────────┼───────────┼────────────────────┤
│ Supervised PCA │ 0.0060         │ top 0.5%  │ 0.4230             │
│                │                │ top 1.0%  │ 0.3427             │
│                │                │ top 2.0%  │ 0.2669             │
│                │                │ top 5.0%  │ 0.2063             │
│                │                │ top 10.0% │ 0.1848             │
├────────────────┼────────────────┼───────────┼────────────────────┤
│ New method     │ 0.2204         │ top 0.5%  │ 0.4053             │
│ (Supervised    │                │ top 1.0%  │ 0.3789             │
│ selection)     │                │ top 2.0%  │ 0.3522             │
│                │                │ top 5.0%  │ 0.3206             │
│                │                │ top 10.0%

This contains both the third and fourth ablation tests: "WSFS-SAF w/o 2020" and "WSFS-SAF w/o Description Length".

In [ ]:
!python WSFS-SAF/ablation/ablation_robustness.py \
    --features-pkl {path} \
    --load-model {path_sm} \
    --top-k 2000 \
    --cov-ds /content/cov_dataset.parquet \
    --original-pairs-ds /content/pairs_dataset.parquet

┌───────────────────┬────────────────┬───────────┬────────────────────┐
│ Method            │ Spearman rho   │ Cutoff    │ Mean correlation   │
├───────────────────┼────────────────┼───────────┼────────────────────┤
│ OOS baseline      │ 0.2204         │ top 0.5%  │ 0.4053             │
│                   │                │ top 1.0%  │ 0.3789             │
│                   │                │ top 2.0%  │ 0.3522             │
│                   │                │ top 5.0%  │ 0.3206             │
│                   │                │ top 10.0% │ 0.2963             │
├───────────────────┼────────────────┼───────────┼────────────────────┤
│ OOS excl. 2020    │ 0.1593         │ top 0.5%  │ 0.2533             │
│                   │                │ top 1.0%  │ 0.2424             │
│                   │                │ top 2.0%  │ 0.2315             │
│                   │                │ top 5.0%  │ 0.2176             │
│                   │                │ top 10.0% │ 0.2055       

# Plots

This section generates the hexbin plot associated with the "WSFS-SAF w/o Description Length" ablation test.

In [ ]:
!python WSFS-SAF/graphs/norm_plot.py \
    --features-pkl {path} \
    --load-model {path_sm} \
    --top-k 2000 \
    --cov-ds /content/cov_dataset.parquet \
    --original-pairs-ds /content/pairs_dataset.parquet \
    --out-dir /content/ # This argument can be commented out if there was no Google Drive mounting. If you would still like to see the graph, then direct it to the Google Colab home directory.


  R²: 0.2714
  Pearson r: 0.5210
  Spearman rho: 0.8014
  Regression: y = 0.230978 * x + -0.6206


# Sandbox / Paper Figures

In [ ]:
!python WSFS-SAF/extract_features.py \
    --features-pkl {path} \
    --top-k 1000 1250 1500 1750 2000 2250 2500 \
    --norm-alpha 0 \
    --score-weight \
    --cov-ds /content/cov_dataset.parquet \
    --original-pairs-ds /content/pairs_dataset.parquet

Scoring features (GPU): 100% 256/256 [01:44<00:00,  2.46it/s]

top-k = 1000

┌───────┬──────────────┬──────────────────────┬──────────────────────┐
│ alpha │ Spearman rho │ Precision-at-k (all) │ Precision-at-k (OOS) │
├───────┼──────────────┼──────────────────────┼──────────────────────┤
│ 0.0   │ 0.2174       │ Top   0.5% = 0.3646  │ Top   0.5% = 0.4025  │
│       │              │ Top   1.0% = 0.3410  │ Top   1.0% = 0.3763  │
│       │              │ Top   2.0% = 0.3188  │ Top   2.0% = 0.3494  │
│       │              │ Top   5.0% = 0.2917  │ Top   5.0% = 0.3162  │
│       │              │ Top  10.0% = 0.2719  │ Top  10.0% = 0.2904  │
└───────┴──────────────┴──────────────────────┴──────────────────────┘

top-k = 1250

┌───────┬──────────────┬──────────────────────┬──────────────────────┐
│ alpha │ Spearman rho │ Precision-at-k (all) │ Precision-at-k (OOS) │
├───────┼──────────────┼──────────────────────┼──────────────────────┤
│ 0.0   │ 0.2182       │ Top   0.5% = 0.3646  │ Top   0.

In [ ]:
!python WSFS-SAF/extract_features.py \
    --features-pkl {path} \
    --top-k 1500 1750 2000 \
    --norm-alpha 0.25 \
    --score-weight \
    --cov-ds /content/cov_dataset.parquet \
    --original-pairs-ds /content/pairs_dataset.parquet

Scoring features (GPU): 100% 256/256 [01:43<00:00,  2.47it/s]

top-k = 1500

┌───────┬──────────────┬──────────────────────┬──────────────────────┐
│ alpha │ Spearman rho │ Precision-at-k (all) │ Precision-at-k (OOS) │
├───────┼──────────────┼──────────────────────┼──────────────────────┤
│ 0.25  │ 0.2191       │ Top   0.5% = 0.3644  │ Top   0.5% = 0.4016  │
│       │              │ Top   1.0% = 0.3394  │ Top   1.0% = 0.3756  │
│       │              │ Top   2.0% = 0.3183  │ Top   2.0% = 0.3484  │
│       │              │ Top   5.0% = 0.2932  │ Top   5.0% = 0.3161  │
│       │              │ Top  10.0% = 0.2737  │ Top  10.0% = 0.2925  │
└───────┴──────────────┴──────────────────────┴──────────────────────┘

top-k = 1750

┌───────┬──────────────┬──────────────────────┬──────────────────────┐
│ alpha │ Spearman rho │ Precision-at-k (all) │ Precision-at-k (OOS) │
├───────┼──────────────┼──────────────────────┼──────────────────────┤
│ 0.25  │ 0.2206       │ Top   0.5% = 0.3638  │ Top   0.

# Interpretability Analysis

In [ ]:
!python WSFS-SAF/interpretability/interpretability_part1.py \
    --features-pkl "$path" --load-model "$path_sm" \
    --cov-ds /content/cov_dataset.parquet \
    --desc-chars 1000


┌────────────────────────┬──────────────────────────────┬──────┬──────┬──────────┬────────────────────────────────────────────────────────────┐
│ Feature                │ Company                      │ Year │ SIC  │ Activ.   │ Description                                                │
├────────────────────────┼──────────────────────────────┼──────┼──────┼──────────┼────────────────────────────────────────────────────────────┤
│ #1: idx 52624          │ POLARITYTE, INC.             │ 2018 │ 2836 │ 14.7     │ Item 1. Business PolarityTE - Welcome to the SHIFT         │
│ s=0.1255               │                              │      │      │          │ PolarityTE Inc., headquartered in Salt Lake City, Utah, is │
│ n=2821                 │                              │      │      │          │ a young and growing commercial-stage, biotechnology        │
│                        │                              │      │      │          │ company founded in 2016 - and we believe the first o

In [ ]:
!python WSFS-SAF/interpretability/interpretability_part2.py \
    --features-pkl "$path" \
    --load-model "$path_sm" \
    --cov-ds /content/cov_dataset.parquet \
    --n-features 20 \
    --top-sics 5


┌───────────┬─────────┬──────────┬───────┬───────┬───────┬─────────┬─────────┬─────┐
│ Feature   │ Score   │ Active   │   SIC │   Obs │   Exp │ Ratio   │ p_adj   │     │
├───────────┼─────────┼──────────┼───────┼───────┼───────┼─────────┼─────────┼─────┤
│ 52624     │ 0.1255  │ 2819     │  2842 │    16 │   1.8 │ 8.7x    │ <0.001  │ *** │
│           │         │          │  8093 │     8 │   1   │ 7.9x    │ <0.001  │ *** │
│           │         │          │  3760 │    19 │   2.6 │ 7.2x    │ <0.001  │ *** │
│           │         │          │  2221 │    18 │   2.8 │ 6.3x    │ <0.001  │ *** │
│           │         │          │  1221 │     6 │   1   │ 5.9x    │ <0.001  │ *** │
├───────────┼─────────┼──────────┼───────┼───────┼───────┼─────────┼─────────┼─────┤
│ 18355     │ 0.1196  │ 4063     │  3442 │    21 │   3.4 │ 6.2x    │ <0.001  │ *** │
│           │         │          │  6399 │    13 │   2.5 │ 5.2x    │ <0.001  │ *** │
│           │         │          │  2013 │    19 │   3.8 │ 5.0x 

# Notes

Interpretability first stage:
- aclanthology.org/2025.acl-industry.73.pdf
- transformer-circuits.pub/2023/monosemantic-features

Second stage:
- pnas.org/doi/10.1073/pnas.2506316122
- hobergphillips.tuck.dartmouth.edu

Third stage:
- arxiv.org/abs/2410.13928
- blog.eleuther.ai/autointerp

# To-Do

0. Trim the literature review. It's kind of bloated just cause we needed to hit a page min for 340w (I'm not sure if this needs to be done right now or after adding the below stuff which is why I'm giving it number 0. We can probably do it throughout)

0. Switch out the LLM and SAE. I'm also giving this number 0 because idk if we even need to do this. Obv this would make our approach more """"novel"""" but idk if that would even matter. we can talk more about this. It also changes the way we do literally everything else below sooooo

3. Further justify correlation-at-k or even change it (if we can't very strongly defend it) since our paper kind of depends on this

5. Add more baselines for comparison (BERT? FinBERT? Vamvourellis et al. embeddings approach?)

6. Add more dim reduction methods for comparison(other than PCA and ours)

7. Interpretability analysis (i.e. take some of the top most informative features and interpret them. Prove that they make sense / are informative)

8. VERY IMPORTANT: do a real case study / actual financial task with our method to show that it works practically (since right now it's very ML-ey)

9. Do more statistical stuff (bootstrap sampling, cross-validation, confidence intervals, all of that. Also more proof that our approach is "sound" i.e. no leakage or whatever)

10. Further justify our scoring method. It's pretty straightforward to us but it might be more convincing if we can prove it has some basis in literature

11. Play around with stuff like taking into account activation for predictive score, etc.